In [1]:
import os
import pandas as pd
import numpy as np
import netCDF4
import matplotlib.pyplot as plt
import numpy as np

from DataFrame_maker import *



In [ ]:
def getNombreBoya(ruta_a_archivo):
    ruta_normalizada = os.path.normpath(ruta_a_archivo)
    partes = ruta_normalizada.split(os.path.sep)
    nombre_boya = partes[-3].replace("-", "_")
    return nombre_boya

def selector_de_archivos(ruta, mensaje, tipo):
    """"Devuelve la ruta al archivo crudo y al archivo validado"""
    archivos = os.listdir(ruta)
    ruta_de_archivos = []
    for archivo in archivos:  
        if f"msj{mensaje}" in archivo and f"_validados_{tipo}_" in archivo and archivo.endswith('.pkl'):
            ruta_de_archivos.append(os.path.normpath(os.path.join(ruta, archivo)))
        
    return ruta_de_archivos

def datenum_to_datetime(datenum):
    import datetime
    """
    Descripción:
        Convierte un array en formato datenum de MATLAB a objetos datetime de Python, redondeando al minuto más cercano y manejando MaskedArray.

    Parámetros:
        datenum (array-like): Array de números datenum (puede ser MaskedArray).

    Retorna:
        np.ndarray: Array de objetos datetime convertidos y redondeados al minuto.

    Ejemplo:
        >>> datenum_to_datetime([738000.501, 738001.499])

    Categoría:
        Funciones-Comunes

    Funciones auxiliares:
        np.array, datetime.datetime.fromordinal, datetime.timedelta
    """
    # Primero convierte el maskedArray en un array normal de python
    if isinstance(datenum, np.ma.MaskedArray):
        fechas = np.array(datenum.filled(np.nan))
    else:
        fechas = np.array(datenum)

    # Lista para almacenar las fechas convertidas
    fechas_convertidas = []
    for idatenum in fechas.flatten():
        # Conversion de idatenum a un valor escalar
        idatenum = float(idatenum)
        # Convierte un número datenum de MATLAB a datetime de Python
        fecha = datetime.datetime.fromordinal(int(idatenum)) + datetime.timedelta(days=idatenum % 1) - datetime.timedelta(days=366)
        # Redondear al minuto más cercano
        fecha = fecha.replace(microsecond=0)
        if fecha.second >= 30:
            fecha += datetime.timedelta(minutes=1)
        fecha = fecha.replace(second=0)
        fechas_convertidas.append(fecha)
    # Retorna las fechas convertidas como un array de numpy
    return np.array(fechas_convertidas)

In [ ]:
# Configuraciones [Acá se indica que carpetas, mensaje y tipo de mensaje se cargará] 
carpetas = ["//192.168.15.249/Med_2025-2026/Reportes_Edit/Reporte_1.3/2026/03. Marzo/BOT1-02-T80/DATOS"] #,
           # "//192.168.15.249/Med_2025-2026/Reportes_Edit/Reporte_1.2/2026/03. Marzo/BOT1-01-T80/DATOS"]
tipos = ["mem"] # realt o mem # Se debe colocar un tipo por cada archivo cargado (esto es para poder comparar una localización de mem contra otra de realt)
msg = 4

In [3]:
# Buscar ruta al archivo validado pkl
rutas_de_archivos = []
for carpeta, tipo in zip(carpetas, tipos):
    rutas = selector_de_archivos(carpeta, msg, tipo)
    rutas_de_archivos.extend(rutas) # Rutas a los archivos: crudo y validado
print(rutas_de_archivos)

['\\\\192.168.15.249\\Med_2025-2026\\Reportes_Edit\\Reporte_1.3\\2026\\03. Marzo\\BOT1-02-T80\\DATOS\\msj4_validados_mem_2025111619_2026020821.pkl']


In [25]:
# Cargar el archivo pkl y crear un dataframe para cada boya
dict = {}
for ruta_de_archivo in rutas_de_archivos:
    df = leer_datos_de_pickle(ruta_de_archivo)
    nombre_boya = getNombreBoya(ruta_de_archivo)
    
    nombre_boya_final = f"{nombre_boya}_validado"
    if "crudo_unido_" in ruta_de_archivo:
        nombre_boya_final = f"{nombre_boya}_crudo" 
    
    if msg == 1: # Viento
        df = crear_dataframe_viento_desde_pickle(df)
    elif msg == 3 or msg == 23: # Oleaje    
        df = crear_dataframe_oleaje_desde_pickle(df)
    elif msg == 4 or msg == 24: # Corrientes
        df, prof = crear_dataframe_corrientes_desde_pickle(df)

In [11]:
# Cargar NC
carpeta_al_nc = "//192.168.15.249/Med_2025-2026/Reportes_Edit/Reporte_1.3/2026/03. Marzo/BOT1-02-T80/RECUPERACIONES/BOT1-02-T80/CORRIENTES/VALIDADOS"
carpeta_al_nc = os.path.normpath(carpeta_al_nc)
ruta_al_nc = os.path.join(carpeta_al_nc, "BOT1-02-T80-SIG500DW-NS103971-Z0.54-MEM-20251116-20260208.nc")
nc = netCDF4.Dataset(ruta_al_nc, 'r')
nc.variables.keys()

dict_keys(['nam', 'Lat', 'Lon', 'TiranteDiseno', 'ProfDiseno', 'TiranteEstimado', 'depth', 'jd', 'Temp', 'u', 'v', 'Heading', 'Pitch', 'Roll', 'ae'])

In [12]:
#Explorar nam
"".join(np.array(nc["nam"][:].flatten()).astype(str))

'BOT1-02-T80-SIG500DW-NS103971-Z0.54-MEM-20251116-20260208'

In [13]:
# Explorar Lat y Lon
lat = nc["Lat"][:]
lon = nc["Lon"][:]
print(f"Latitud: {lat}, Longitud: {lon}")
# Explorar tirantes y profundidades
TiranteDiseno = nc["TiranteDiseno"][:]
ProfDiseno = nc["ProfDiseno"][:]
TiranteEstimado = nc["TiranteEstimado"][:]
print(f"Tirante de Diseño: {TiranteDiseno}, Profundidad de Diseño: {ProfDiseno}, Tirante Estimado: {TiranteEstimado}")

Latitud: [19.085995], Longitud: [-92.72543]
Tirante de Diseño: [80.], Profundidad de Diseño: [0.54], Tirante Estimado: [84.]


In [14]:
# # explorar grados y frecuencia
# grados = nc["grados"][:]
# frec = nc["frec"][:]
# print(f"Grados: {grados}, Frecuencia: {frec}")

In [15]:
# Explorar fechas del netCDF (Vienen en formato datenum de MATLAB)
jd = np.array(nc["jd"][:].flatten())
# Obtener jd del netCDF
fechas = datenum_to_datetime(jd)    
print(f"fecha inicial = {fechas[0]}, fecha final = {fechas[-1]}")

fecha inicial = 2025-11-16 19:00:00, fecha final = 2026-02-08 21:00:00


In [16]:
print(nc.variables.keys()) # Variables del archivo nc
print(df.columns) # Variables del archivo pkl

dict_keys(['nam', 'Lat', 'Lon', 'TiranteDiseno', 'ProfDiseno', 'TiranteEstimado', 'depth', 'jd', 'Temp', 'u', 'v', 'Heading', 'Pitch', 'Roll', 'ae'])
Index(['tspan', 'rap_3', 'dir_3', 'rap_7', 'dir_7', 'rap_11', 'dir_11',
       'rap_15', 'dir_15', 'rap_19', 'dir_19', 'rap_23', 'dir_23', 'rap_27',
       'dir_27', 'rap_31', 'dir_31', 'rap_35', 'dir_35', 'rap_39', 'dir_39',
       'rap_43', 'dir_43', 'rap_47', 'dir_47', 'Temp_ADCP'],
      dtype='object')


In [17]:
# nc.variables["i_uvrec"] = np.nan

In [12]:
# fecha_idx = 0  # Cambia este valor para otra fecha

# # Extrae la matriz 2D para esa fecha
# matriz_2d = dirspec[fecha_idx, :, :]

# plt.figure(figsize=(8, 6))
# plt.imshow(matriz_2d, aspect='auto', origin='lower', cmap='viridis')
# plt.colorbar(label='Valor')
# plt.title(f'Datos para fecha índice {fecha_idx}')
# plt.xlabel('ifrecuencia')
# plt.ylabel('igrados')
# plt.show()

In [ ]:
# Comparar Hs del nc con el Hs del df
var_nc_name = "Temp"
var_df_name = "Temp_ADCP"
var_nc = np.array(nc[var_nc_name][:].flatten()).astype(np.float32)
var_df = df[var_df_name].values.flatten().astype(np.float32)
count = 0
for i in range(len(var_nc)):
    
    if var_nc[i] == var_df[i]:
        count += 1
        
cantidad_nan = np.isnan(var_nc).sum()
cantidad_validados = len(var_nc)-cantidad_nan
porcentaje_validados = cantidad_validados/len(var_nc)*100

print(f"Cantidad de datos = {len(var_nc)}, %validados = {porcentaje_validados:.2f}%, Variable = {var_nc_name}, Cantidad de datos iguales = {count}, Porcentaje de datos iguales = {count/len(var_nc)*100:.2f}%")

Cantidad de datos = 2019, %validados = 100.00%, Variable = Temp, Cantidad de datos iguales = 2019, Porcentaje de datos iguales = 100.00%


In [ ]:
var_nc = np.array(nc["u"][:]).astype(np.float32)
u1 = var_nc[:,0].astype(np.float32)


array([ 0.0284, -0.097 , -0.1241, ..., -0.0591, -0.0483, -0.0446],
      shape=(2019,), dtype=float32)

In [ ]:
# dirspec = np.array(nc.variables["DirSpec"][:])
# dirspec.shape


In [ ]:
# fecha_idx = 0  # Cambia este valor para otra fecha

# # Extrae la matriz 2D para esa fecha
# matriz_2d = dirspec[fecha_idx, :, :]

# plt.figure(figsize=(8, 6))
# plt.imshow(matriz_2d, aspect='auto', origin='lower', cmap='viridis')
# plt.colorbar(label='Valor')
# plt.title(f'Datos para fecha índice {fecha_idx}')
# plt.xlabel('ifrecuencia')
# plt.ylabel('igrados')
# plt.show()

In [ ]:
%matplotlib inline

In [14]:
ruta = "//192.168.15.249/Med_2025-2026/Reportes_Edit/Reporte_1.3/2026/03. Marzo/BOT1-02-T80/RECUPERACIONES/BOT1-02-T80/CORRIENTES/VALIDADOS"
archivo = "BOT1-02-T80-SIG500DW-NS103971-Z0.54-MEM-20251116-20260208.nc"

ruta_archivo = os.path.normpath(os.path.join(ruta, archivo))

var_to_remove = 'dim_ae'

with netCDF4.Dataset(ruta_archivo, 'r') as src, netCDF4.Dataset(archivo, 'w') as dst:
    # Copiar atributos globales
    dst.setncatts(src.__dict__)

    # Copiar dimensiones
    for name, dimension in src.dimensions.items():
        dst.createDimension(
            name, (len(dimension) if not dimension.isunlimited() else None))

    # Copiar variables excepto la que queremos eliminar
    for name, variable in src.variables.items():
        if name != var_to_remove:
            x = dst.createVariable(
                name, variable.datatype, variable.dimensions)
            dst.variables[name].setncatts(src.variables[name].__dict__)
            dst.variables[name][:] = src.variables[name][:]

In [13]:
nc_new = netCDF4.Dataset(archivo, 'r')
nc_new.variables.keys()


NameError: name 'archivo' is not defined

In [ ]:
nc_new.variables["i_uvrec"]

In [ ]:
nc_new.close()

In [ ]:
nc.variables["ae"]